In [ ]:
import os

os.environ["KERAS_BACKEND"] = "torch"
os.environ["CUDA_VISIBLE_DEVICES"] = "1"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import keras
import torch
import numpy as np
import albumentations as A

from pathlib import Path
from typing import Sequence

from PIL import Image
from torch.utils.data import Dataset, DataLoader

from agx_core.transforms import (
    LogTransform,
    GammaCorrection,
    Deskew,
)

keras.utils.set_random_seed(42)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False


class UnlabeledImageDataset(Dataset):
    def __init__(self, root_dir: Path, cond_shape: Sequence[int], transform=None):
        self.root_dir = root_dir
        self.transform = transform
        self.cond_shape = cond_shape
        # Get list of all image file names in the folder
        self.image_files = list(root_dir.glob("*.bmp"))

    def __len__(self):
        return len(self.image_files)

    def __getitem__(self, idx):
        img_name = self.image_files[idx]
        image = Image.open(img_name).convert("L")

        if self.transform:
            image = self.transform(image=np.array(image))
            image = image["image"][..., np.newaxis]

        condition = np.ones(self.cond_shape, dtype=np.float32)

        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        image = torch.tensor(image, device=device)
        return (image, torch.tensor(condition, device=device)), image


def train_transforms(img_size, mean=[0.5], std=[0.5]):
    return A.Compose(
        [
            Deskew(),
            A.Pad((25, 25), 255, p=0.25),
            A.Pad((50, 50), 255, p=0.25),
            A.Pad((75, 75), 255, p=0.25),
            LogTransform(epsilon=1, invert=True),
            GammaCorrection(center=0.9, gain=9),
            A.Resize(img_size, img_size),
            A.Affine(scale=(0.9, 0.95), rotate=(-90, 90), shear=(5, 5), p=0.5),
            A.RandomRotate90(0.5),
            A.GaussianBlur(blur_range=(1, 3), p=0.3),
            A.Normalize(mean=mean, std=std),
        ]
    )


def valid_transforms(img_size, mean=[0.5], std=[0.5]):
    return A.Compose(
        [
            Deskew(),
            LogTransform(epsilon=1, invert=True),
            GammaCorrection(center=0.9, gain=9),
            A.Resize(img_size, img_size),
            A.Normalize(mean=mean, std=std),
        ]
    )

In [ ]:
img_size = 224
res = img_size // 2**5

img_shape = (None, img_size, img_size, 1)
cond_shape = (None, res, res, 1)

In [ ]:
train_path = Path("../data/products/LaTuaPastaGlassJars/Clean/train/")
valid_path = Path("../data/products/LaTuaPastaGlassJars/Clean/val/")
test_path = Path("../data/products/LaTuaPastaGlassJars/Test/images/train/")

ds_train = UnlabeledImageDataset(
    train_path, transform=train_transforms(img_size), cond_shape=cond_shape[1:]
)
ds_valid = UnlabeledImageDataset(
    valid_path, transform=valid_transforms(img_size), cond_shape=cond_shape[1:]
)
ds_test = UnlabeledImageDataset(
    test_path, transform=valid_transforms(img_size), cond_shape=cond_shape[1:]
)

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(3, 3, figsize=(12, 12))

for idx, ax in enumerate(axes.flat):
    X, y = ds_train[idx]
    image = X[0].cpu().numpy()
    ax.imshow(image, cmap="gray")
    ax.set_title(f"Image {idx + 1}")
    ax.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
from keras.optimizers import Adam

from agx_core.models.reversed_autoencoder import (
    MobileNetV3SmallEncoder,
    MobileNetV3SmallDecoder,
)
from agx_torch.models.reversed_autoencoder.model import ReversedAutoencoder

spatial_temp = 3.0
enc = MobileNetV3SmallEncoder(latent_size=512, weights="imagenet")
dec = MobileNetV3SmallDecoder(target_shape=img_shape[1:])
ra = ReversedAutoencoder(enc, dec, beta_kld=0.7, spatial_temp=spatial_temp)

ra.build([img_shape, cond_shape])
ra.compile(Adam(learning_rate=1e-6), Adam(learning_rate=1e-3))

ra.summary()

loader_train = DataLoader(ds_train, batch_size=16, shuffle=True)
loader_valid = DataLoader(ds_valid, batch_size=16)

In [ ]:
import matplotlib.pyplot as plt
from typing import Optional, Dict, Any

rec_dir = Path("reconstructions")
rec_dir.mkdir(exist_ok=True)

plot_every = 20
rec_test_samples = 10
samples = []
for idx in range(3):
    sample = keras.ops.concatenate(
        [
            ds_train[idx][-1][None, ...],
            ds_valid[idx][-1][None, ...],
            ds_test[idx][-1][None, ...],
        ]
    )
    samples.append(sample)

samples = keras.ops.concatenate(samples)

conds = ds_train[idx][0][1][None, ...]
single_cond = conds.clone()
conds = keras.ops.tile(conds, 9)
conds = keras.ops.transpose(conds, (3, 2, 1, 0))

import numpy as np

x = np.linspace(-3, 3, 100)
standard_normal = (1 / np.sqrt(2 * np.pi)) * np.exp(-0.5 * x**2)


def plot_on_step_end(step: int, logs: Optional[Dict[str, Any]] = None):
    # plot reconstruction every n epochs/batches
    if (step + 1) % plot_every != 0:
        return

    with torch.no_grad():
        samples_resized = ra.resize_progressive_output(samples)

        mean_logvar, _ = ra.encoder([samples_resized, conds], training=False)
        z = ra.reparameterize(mean_logvar, training=False)
        reconstructed = ra.decoder([z, conds], training=False)

        error = ra.reconstruction_loss(samples_resized, reconstructed)

    samples_resized = samples_resized.cpu().numpy()
    reconstructed = reconstructed.cpu().numpy()
    error = error.cpu().numpy()
    z_np = z.cpu().numpy()

    height, width = samples_resized.shape[1:3]
    ar = width / height

    figwidth = 4
    figheight = figwidth / ar

    n_samples = samples.shape[0]

    # 4 columns: original, reconstructed, difference, latent histogram
    fig, axs = plt.subplots(
        n_samples,
        4,
        figsize=(4 * figwidth, n_samples * figheight),
    )

    # Handle single row case
    if n_samples == 1:
        axs = axs.reshape(1, -1)

    for row, (real, rec, err, z_sample) in enumerate(
        zip(samples_resized, reconstructed, error, z_np)
    ):

        if row >= n_samples:
            break

        if row == 0:
            axs[row, 0].set_title("Original", fontsize=12, pad=15)
            axs[row, 1].set_title("Reconstruction", fontsize=12, pad=15)
            axs[row, 2].set_title("Anomaly Map", fontsize=12, pad=15)
            axs[row, 3].set_title("Latent Distribution", fontsize=12, pad=15)

        # First column: keep axis on for ylabel, hide ticks, rotate 90 degrees
        axs[row, 0].set_ylabel(
            f"{row+1}",
            fontsize=10,
            rotation=90,
            labelpad=15,
            ha="center",
            va="center",
        )

        # Image columns: hide ticks
        for col in range(3):
            axs[row, col].set_xticks([])
            axs[row, col].set_yticks([])

        axs[row, 0].imshow(real, cmap="gray")
        axs[row, 1].imshow(rec, cmap="gray")
        axs[row, 2].imshow(err, cmap="inferno")

        # Plot histogram of latent variables
        axs[row, 3].hist(z_sample.flatten(), bins=50, alpha=0.7, density=True)
        axs[row, 3].set_xlabel("z value")
        axs[row, 3].set_ylabel("Density")
        axs[row, 3].set_xlim(left=-4, right=4)
        axs[row, 3].set_ylim(bottom=0, top=1.0)

        # Overlay standard normal for comparison
        axs[row, 3].plot(x, standard_normal, "r--", alpha=0.8, label="N(0,1)")
        axs[row, 3].legend()

    epoch_num = step + 1
    filename = f"epoch_{epoch_num:05d}.jpg"
    title = f"Reconstruction Results - Epoch {epoch_num}"

    # Add overall figure title
    fig.suptitle(title, fontsize=14, y=0.96)

    plt.tight_layout()
    plt.subplots_adjust(top=0.92, left=0.08)

    fig.savefig(rec_dir / filename)

    plt.close(fig)

    print(f"Epoch {step + 1}: reconstruction results saved")

In [ ]:
from keras.callbacks import ModelCheckpoint, LambdaCallback

from agx_torch.callbacks import (
    GarbageCollectionCallback,
    MemorySnapshotCallback,
    MemoryLeakDiagnosticCallback,
)
from agx_core.models.reversed_autoencoder.callbacks import (
    AdversarialEquilibriumCallback,
    ProgressiveGrowingCallback,
    BackboneThawCallback,
)

callbacks = [
    LambdaCallback(on_epoch_end=plot_on_step_end),
    AdversarialEquilibriumCallback(0.5, -0.5, min_pause_steps=100),
    # ProgressiveGrowingCallback(3000, 3000),
    BackboneThawCallback(patience=500),
    ModelCheckpoint(
        filepath="ra_mbnetv3.best.keras",
        monitor="val_loss_rec",
        mode="min",
        save_best_only=True,
        verbose=1,
    ),
    # MemorySnapshotCallback((130, 260, 650), 26),
    # MemoryLeakDiagnosticCallback(39, 13, 130),
    # GarbageCollectionCallback(130, 195, True),
]

In [ ]:
history = ra.fit(
    loader_train,
    validation_data=loader_valid,
    initial_epoch=0,
    epochs=51,
    callbacks=callbacks,
    verbose=0,
)

In [ ]:
ra.save("ra_mbnetv3.model.keras")

In [ ]:
import pandas as pd

df = pd.DataFrame.from_dict(history.history)

hist_file = Path("history.csv")
if hist_file.exists():
    hist = pd.read_csv(hist_file)
    df.index += len(hist)
    hist = pd.concat([hist, df])
    # hist.to_csv(hist_file, index=False)
else:
    df.to_csv(hist_file, index=False)
    hist = df
hist.tail()

In [ ]:
loader_test = DataLoader(ds_test, batch_size=16)

In [ ]:
import cv2
from keras import ops, models
import matplotlib.pyplot as plt


from agx_core.models.reversed_autoencoder.model import kl_divergence
from agx_core.models.reversed_autoencoder.anomaly import anomaly_pipeline
from agx_core.models.reversed_autoencoder.anomaly_viz import AnomalyExplorer

ra_best = ra  # models.load_model("ra_mbnetv3.best.keras")
explorer = AnomalyExplorer(
    model=ra_best,
    dataloader=loader_test,
    n_samples=9,
    device="cpu",
    # use_feature_maps=True,
)
explorer.show()

In [ ]:
import pandas as pd

fig, ax = plt.subplots(figsize=(21, 14))

# Adjust val_loss_embed, forgot to update validation scaling
hist[["elbo_real", "kld_real"]].plot(ax=ax)

In [ ]:
from agx_core.models.reversed_autoencoder.model import kl_divergence

with torch.device("cpu"):
    ra.eval()
    I = torch.rand((1, *img_shape[1:]))
    C = torch.rand((1, *cond_shape[1:]))
    Z = torch.rand((1, *cond_shape[1:3], 512))

    import keras

    i_ = keras.Input(img_shape[1:])
    c_ = keras.Input(cond_shape[1:])

    (mean, logvar), _ = ra.encoder([i_, c_])
    z = ra.reparameterize([mean, logvar])
    rec = ra.decoder([z, c_])
    kld = kl_divergence(mean, logvar)

    model = keras.Model(
        inputs=[i_, c_], outputs=[rec, kld], name="reversed_autoencoder"
    )
    model.training = False
    model.eval()

    torch.onnx.export(
        model,
        ([I, C],),
        "model.onnx",
        input_names=["image", "condition"],
        output_names=["reconstruction", "kld"],
    )

In [ ]:
import matplotlib.pyplot as plt

from agx_core.helpers import _channel_axis
from keras import ops


def kl_divergence(mean, logvar):
    return 0.5 * ops.sum(
        ops.square(mean) + ops.exp(logvar) - logvar - 1.0,
        axis=_channel_axis(),
    )


fig, axes = plt.subplots(5, 3, figsize=(12, 20))

for idx in range(5):
    (I, C), y = ds_test[idx]
    I = I.cpu().detach().numpy()
    rec = ra([I[np.newaxis, ...], C[np.newaxis, ...]]).cpu().detach().numpy()[0]
    D = mse_weighted(I[np.newaxis, ...], rec[np.newaxis, ...], 7).cpu().detach().numpy()[0]
    
    (mean, logvar), _ = ra.encoder([I[np.newaxis, ...], C[np.newaxis, ...]])
    kld = kl_divergence(mean, logvar).cpu().detach().numpy()[0]
    
    axes.flat[3 * idx].imshow(rec, cmap="gray")
    axes.flat[3 * idx + 1].imshow(I, cmap="gray")
    axes.flat[3 * idx + 2].imshow(kld, cmap="inferno")
    axes.flat[3 * idx].axis("off")
    axes.flat[3 * idx + 1].axis("off")
    axes.flat[3 * idx + 2].axis("off")

plt.tight_layout()
plt.show()